In [1]:
import json
import os
import sys
import time
from datetime import date, datetime

import numpy as np
import pandas as pd
import polars as pl
import pyarrow

In [2]:
data_2019 = "../../novus/matchingnemo/scratch/safegraph_data/Weekly Patterns/2019_Weekly_Patterns/"
example = os.listdir(data_2019)[2]

In [3]:
cols_to_read = [
    "safegraph_place_id",
    "location_name",
    "street_address",
    "city",
    "region",
    "postal_code",
    "iso_country_code",
    "date_range_start",
    "raw_visit_counts",
    "raw_visitor_counts",
    "visits_by_day",
    "visits_by_each_hour",
    "poi_cbg",
    "visitor_home_cbgs",
    "visitor_daytime_cbgs",
    "visitor_country_of_origin",
    "distance_from_home",
    "median_dwell",
    "bucketed_dwell_times",
]
schema_overrides = {"date_range_start": pl.Datetime, "distance_from_home": pl.Int64}

In [4]:
read = (
    pl.scan_csv(os.path.join(data_2019, example), schema_overrides=schema_overrides)
    .select(cols_to_read)
    .with_columns(
        [
            pl.col("visits_by_each_hour").str.json_decode(pl.List(pl.Int64)),
            pl.col("visits_by_day").str.json_decode(pl.List(pl.Int64)),
        ]
    )
)

In [5]:
cols_to_select = [
    "safegraph_place_id",
    "city",
    "region",
    "date_range_start",
    "visits_by_each_hour",
]

In [6]:
read = (
    read.filter(
        pl.col("iso_country_code") == "US",
        pl.col("city") == "Portland",
        pl.col("region") == "OR",
    )
    .select(cols_to_select)
    .with_columns(t=(pl.int_ranges(0, pl.col("visits_by_each_hour").list.len())))
    .explode(["t", "visits_by_each_hour"])
    .rename({"visits_by_each_hour": "visits"})
)

In [7]:
read = read.collect()

In [8]:
min(read['date_range_start'].dt.week().to_numpy())

29

In [9]:
read.head()

safegraph_place_id,city,region,date_range_start,visits,t
str,str,str,datetime[μs],i64,i64
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,1,0
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,1
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,2
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,3
"""sg:05b435392f714335858efe05970…","""Portland""","""OR""",2019-07-15 07:00:00,0,4


<b> Augment with FEMA data

In [10]:
!pip install shapely geopandas

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_thunder-0.2.0.dev0-py3.12.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.23a0+6627725-py3.12-linux-aarch64.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.12.0.dev0-py3.12.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/loosevers

In [11]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [12]:
target_folder = r"../../novus/matchingnemo/scratch/safegraph_data/Weekly Patterns/Digital_Twins_Analysis/temporary_stash_very_heavy"
core_places_data_2019 = "../../novus/matchingnemo/scratch/safegraph_data/Core Places Data/CoreRecords-CORE_POI-2019_03-2020-03-25"
city = "Portland"

In [13]:
def find_pois_contained_in_fema_geometry(fema_data, safegraph_place_data):
    # given a fema dataset and a safegraph dataset, loop through the sg pois and see what fema build id it"s within, then write the poi id to fema dataset as match
    places_city = safegraph_place_data[safegraph_place_data["city"] == city]
    sg_match = ["" for _ in range(len(fema_data))]
    for poi in places_city.itertuples(index=False):
        poi_point = Point(getattr(poi, "longitude"), getattr(poi, "latitude"))
        poi_id = getattr(poi, "safegraph_place_id")
        fema_data["contains"] = fema_data["geometry"].contains(poi_point)
        match_indices = fema_data[fema_data["contains"] == True].index
        if match_indices.size == 0:
            continue
        match_ind = np.random.choice(match_indices)
        sg_match[match_ind] = poi_id
    fema_data["sg_match"] = sg_match

    return fema_data

In [14]:
gdf = gpd.read_file(r"../../novus/matchingnemo/scratch/safegraph_data/Weekly Patterns/Digital_Twins_Analysis/temporary_stash_very_heavy/entire_or_structures_clip.gpkg")
gdf_subset = gdf[
    [
        "BUILD_ID",
        "OCC_CLS",
        "PRIM_OCC",
        "SQMETERS",
        "SQFEET",
        "CENSUSCODE",
        "UUID",
        "geometry",
    ]
]
gdf_nonresidential = gdf_subset[gdf_subset["OCC_CLS"] != "Residential"]


places_data = pd.read_csv(
    r"../../novus/matchingnemo/scratch/Safegraph/Digital_Twins_Analysis/temporary_stash_very_heavy/2019_Portland.csv"
)
places_centroid = gpd.GeoDataFrame(
    places_data,
    geometry=gpd.points_from_xy(places_data.longitude, places_data.latitude),
    crs="EPSG:4326",
)

In [15]:
matches = gpd.sjoin(
    gdf_nonresidential, places_centroid, predicate="contains", how="left"
)

In [16]:
cols_in_csv = [
    "BUILD_ID",
    "OCC_CLS",
    "PRIM_OCC",
    "SQMETERS",
    "SQFEET",
    "CENSUSCODE",
    "UUID",
    "safegraph_place_id",
]

In [17]:
read_pd = read.to_pandas()
matches["safegraph_place_id"] = matches["safegraph_place_id"].astype(str)
read_pd["safegraph_place_id"] = read_pd["safegraph_place_id"].astype(str)
outfile = matches.merge(read_pd, on="safegraph_place_id", how="left")

In [23]:
cols_in_csv = [
    "BUILD_ID",
    "OCC_CLS",
    "PRIM_OCC",
    "SQMETERS",
    "SQFEET",
    "CENSUSCODE",
    "UUID",
    "safegraph_place_id",
    "date_range_start",
    "t",
    "visits"
]
outfile = outfile.loc[:,cols_in_csv]

outfile2 = outfile.groupby(['safegraph_place_id', 't']).agg({'visits': 'sum'}).reset_index()


,BUILD_ID,OCC_CLS,PRIM_OCC,SQMETERS,SQFEET,CENSUSCODE,UUID,safegraph_place_id,date_range_start,t,visits
1,2350928,Unclassified,Unclassified,67.231216,723.670105,41025960200,{38b01679-552a-4a84-936b-58811561c36b},nan,NaT,NaN,NaN
2,2350935,Agriculture,Agriculture,235.860687,2538.780762,41025960200,{e8501fa8-b19b-440b-8bf8-f90fd90f08ac},nan,NaT,NaN,NaN
3,2350993,Unclassified,Unclassified,308.850983,3324.441162,41025960200,{5f540e3d-e257-4371-a60e-f8b0bec87ec6},nan,NaT,NaN,NaN
4,2351003,Agriculture,Agriculture,61.829311,665.524536,41025960200,{c4127cbc-5d80-4aee-b7a3-c95e61223f13},nan,NaT,NaN,NaN
5,2351005,Agriculture,Agriculture,82.150963,884.264771,41025960200,{2502a930-399f-426e-98fe-26202072d5f5},nan,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1862767,539915,Unclassified,Unclassified,243.317795,2619.048340,41051007202,{5d436060-6bed-4153-a5b2-a474a95f0120},nan,NaT,NaN,NaN
1862768,593463,Unclassified,Unclassified,60.114212,647.063354,41051007100,{7f48325e-c1e9-49ae-8688-50408f7c475d},nan,NaT,NaN,NaN
1862769,855018,Industrial,Light,395.033020,4252.096191,41051007202,{973c03f6-612a-4ff5-b8ac-9bb3383b7a33},nan,NaT,NaN,NaN
1862770,855019,Industrial,Light,618.722961,6659.872070,41051007202,{d0f19ae0-e4cc-4b0a-bbf4-5a1051e2ab92},nan,NaT,NaN,NaN


In [ ]:
# outfile.write_csv("../../novus/matchingnemo/scratch/ampnet_data/test.csv")